In [3]:
import pandas as pd 
import numpy as np
import os

In [4]:
os.chdir("/Users/kaiping/Desktop/olist_project/data") 
os.getcwd()

'/Users/kaiping/Desktop/olist_project/data'

In [6]:
df_orders      = pd.read_csv("raw/olist_orders_dataset.csv")
df_order_reviews  = pd.read_csv("raw/olist_order_reviews_dataset.csv")
df_customers      = pd.read_csv("raw/olist_customers_dataset.csv")

In [7]:
# 複製所需要表
orders = df_orders.copy()
customers = df_customers.copy()
reviews = df_order_reviews.copy()

In [8]:
# 時間欄位轉 datetime
date_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [10]:
print(orders.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                        object
order_delivered_carrier_date             object
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [11]:
# 只保留 status = delivered 訂單
orders_delivered = orders[orders["order_status"] == "delivered"].copy()

In [25]:
# 一筆訂單可以有多個評論? 
# 每個 order_id 有幾筆 review
review_per_order = (
    reviews
    .groupby("order_id")
    .agg(
        review_cnt=("review_id", "count"),
        review_id_nunique=("review_id", "nunique"),
        review_score_mean=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_score_max=("review_score", "max")
    )
    .reset_index()
)

# 檢查是否存在一筆 order 多筆 review
review_per_order["multi_review_flag"] = review_per_order["review_cnt"] > 1

review_per_order["multi_review_flag"].value_counts()

multi_review_flag
False    98126
True       547
Name: count, dtype: int64

發現少數 order_id 對應多筆 review 紀錄（約 0.55%），  
若直接 merge 會造成 order grain 被重複展開。  

因此先將 review 以 order_id 聚合為單一 order-level review_score_mean，  
再 merge 回 order_event_base，
以維持「一筆 order = 一列」的 grain 一致性。

In [29]:
# review 聚合到 order grain 
review_order_agg = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score_mean=("review_score", "mean")
    )
)


review_order_agg

,order_id,review_score_mean
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0
...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.0
98669,fffcd46ef2263f404302a634eb57f7eb,5.0
98670,fffce4705a9662cd70adb13d4a31832d,5.0
98671,fffe18544ffabc95dfada21779c9644f,5.0


In [32]:
# orders + customers 

# orders + customers

order_event_base = orders_delivered.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

In [35]:
order_event_base.isna().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
customer_unique_id                0
customer_state                    0
dtype: int64

In [36]:
# merge review_score_mean 
order_event_base = order_event_base.merge(
    review_order_agg,
    on = "order_id",
    how = "left"
)

In [40]:
# 保留所需要欄位
order_event_base = order_event_base[
    [
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "customer_state",
        "review_score_mean"
    ]
].copy()

order_event_base

,order_id,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_state,review_score_mean
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:00,2017-10-10 21:25:00,2017-10-18,SP,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:00,2018-08-07 15:27:00,2018-08-13,BA,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:00,2018-08-17 18:06:00,2018-09-04,GO,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:00,2017-12-02 00:28:00,2017-12-15,RN,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:00,2018-02-16 18:17:00,2018-02-26,SP,5.0
...,...,...,...,...,...,...,...
96473,9c5dedf39a927c1b2549525ed64a053c,6359f309b166b0196dbf7ad2ac62bb5a,2017-03-09 09:54:00,2017-03-17 15:08:00,2017-03-28,SP,5.0
96474,63943bddc261676b46f01ca7ac2f7bd8,da62f9e57a76d978d02ab5362c509660,2018-02-06 12:58:00,2018-02-28 17:37:00,2018-03-02,SP,4.0
96475,83c1379a015df1e13d02aae0204711ab,737520a9aad80b3fbbdad19b66b37b30,2017-08-27 14:46:00,2017-09-21 11:24:00,2017-09-27,BA,5.0
96476,11c177c8e97725db2631073c19f07b62,5097a5312c8b157bb7be58ae360ef43c,2018-01-08 21:28:00,2018-01-25 23:32:00,2018-02-15,RJ,2.0


In [41]:
order_event_base.isna().sum()

order_id                           0
customer_unique_id                 0
order_purchase_timestamp           0
order_delivered_customer_date      8
order_estimated_delivery_date      0
customer_state                     0
review_score_mean                646
dtype: int64

In [42]:
# order_delivered_customer_date 缺失刪除

order_event_base = order_event_base[
    order_event_base["order_delivered_customer_date"].notna()
].copy()
order_event_base.isna().sum()

order_id                           0
customer_unique_id                 0
order_purchase_timestamp           0
order_delivered_customer_date      0
order_estimated_delivery_date      0
customer_state                     0
review_score_mean                646
dtype: int64

In [43]:
# review_score_mean 缺失訊號保留，後續建模處理

In [ ]:
# 建立延伸欄位

In [44]:
# delivery_days = order_delivered_customer_date - order_purchase_timestamp 
order_event_base["delivery_days"] = (
    order_event_base["order_delivered_customer_date"]
    - order_event_base["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

order_event_base

,order_id,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_state,review_score_mean,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:00,2017-10-10 21:25:00,2017-10-18,SP,4.0,8.436806
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:00,2018-08-07 15:27:00,2018-08-13,BA,4.0,13.781944
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:00,2018-08-17 18:06:00,2018-09-04,GO,5.0,9.394444
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:00,2017-12-02 00:28:00,2017-12-15,RN,5.0,13.208333
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:00,2018-02-16 18:17:00,2018-02-26,SP,5.0,2.874306
...,...,...,...,...,...,...,...,...
96473,9c5dedf39a927c1b2549525ed64a053c,6359f309b166b0196dbf7ad2ac62bb5a,2017-03-09 09:54:00,2017-03-17 15:08:00,2017-03-28,SP,5.0,8.218056
96474,63943bddc261676b46f01ca7ac2f7bd8,da62f9e57a76d978d02ab5362c509660,2018-02-06 12:58:00,2018-02-28 17:37:00,2018-03-02,SP,4.0,22.193750
96475,83c1379a015df1e13d02aae0204711ab,737520a9aad80b3fbbdad19b66b37b30,2017-08-27 14:46:00,2017-09-21 11:24:00,2017-09-27,BA,5.0,24.859722
96476,11c177c8e97725db2631073c19f07b62,5097a5312c8b157bb7be58ae360ef43c,2018-01-08 21:28:00,2018-01-25 23:32:00,2018-02-15,RJ,2.0,17.086111


In [46]:
# 是否延遲 late_flag = order_delivered_customer_date > order_estimated_delivery_date

order_event_base["late_flag"] = (
    order_event_base["order_delivered_customer_date"]
    > order_event_base["order_estimated_delivery_date"]
).astype(int)

order_event_base["late_flag"].value_counts()

late_flag
0    88644
1     7826
Name: count, dtype: int64

In [47]:
# 是否有評論
order_event_base["has_review"] = (
    order_event_base["review_score_mean"].notna()
).astype(int)

order_event_base["has_review"].value_counts()

has_review
1    95824
0      646
Name: count, dtype: int64

In [48]:
# 合併每個customer_unique_id同 timestamp orders → purchase_event_base
# 同一位顧客，在同一個 purchase timestamp 的多筆 order，視為一次購買事件
# grain: 每個customer_unique_id 的每個purchase_event

purchase_event_base = (
    order_event_base
    .groupby(
        [
            "customer_unique_id",
            "order_purchase_timestamp"
        ],
        as_index=False
    )
    .agg(
        delivery_days_mean=("delivery_days", "mean"),
        first_event_review_score_mean=("review_score_mean", "mean"),
        late_flag=("late_flag", "max"),
        has_review=("has_review", "max"),
        customer_state=("customer_state", "first")
    )
)
purchase_event_base

,customer_unique_id,order_purchase_timestamp,delivery_days_mean,first_event_review_score_mean,late_flag,has_review,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,6.411111,5.0,0,1,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:00,3.285417,4.0,0,1,SP
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,25.731250,3.0,0,1,SC
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:00,20.037500,4.0,0,1,PA
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,13.140972,5.0,0,1,SP
...,...,...,...,...,...,...,...
95758,fffcf5a5ff07b0908bd4e2dbc735a684,2017-06-08 21:00:00,27.515278,5.0,0,1,PE
95759,fffea47cd6d3cc0a88bd621562a9d061,2017-12-10 20:07:00,30.097917,4.0,0,1,BA
95760,ffff371b4d645b6ecea244b27531430a,2017-02-07 15:49:00,14.872222,5.0,0,1,MT
95761,ffff5962728ec6157033ef9805bacc48,2018-05-02 15:17:00,11.859028,5.0,0,1,ES


In [52]:
# 依據customer_unique_id + purchase_timestamp排序
purchase_event_base = (
    purchase_event_base
    .sort_values(
        [
            "customer_unique_id",
            "order_purchase_timestamp"
        ]
    )
    .copy()
)

In [53]:
# 計算每個cutomer 的每筆訂單的rank (第一筆訂單1,第二筆2 .....) 
purchase_event_base["purchase_event_rank"] = (
    purchase_event_base
    .groupby("customer_unique_id")
    .cumcount()
    + 1
)
purchase_event_base

,customer_unique_id,order_purchase_timestamp,delivery_days_mean,first_event_review_score_mean,late_flag,has_review,customer_state,purchase_event_rank
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,6.411111,5.0,0,1,SP,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:00,3.285417,4.0,0,1,SP,1
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,25.731250,3.0,0,1,SC,1
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:00,20.037500,4.0,0,1,PA,1
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,13.140972,5.0,0,1,SP,1
...,...,...,...,...,...,...,...,...
95758,fffcf5a5ff07b0908bd4e2dbc735a684,2017-06-08 21:00:00,27.515278,5.0,0,1,PE,1
95759,fffea47cd6d3cc0a88bd621562a9d061,2017-12-10 20:07:00,30.097917,4.0,0,1,BA,1
95760,ffff371b4d645b6ecea244b27531430a,2017-02-07 15:49:00,14.872222,5.0,0,1,MT,1
95761,ffff5962728ec6157033ef9805bacc48,2018-05-02 15:17:00,11.859028,5.0,0,1,ES,1


In [60]:
# 找 first_purchase_ts
first_purchase = (
    purchase_event_base[
        purchase_event_base["purchase_event_rank"] == 1
    ]
    .rename(
        columns={
            "order_purchase_timestamp": "first_purchase_ts"
        }
    )
    .copy()
)

In [61]:
# 找 second_purchase_ts
second_purchase = (
    purchase_event_base[
        purchase_event_base["purchase_event_rank"] == 2
    ][
        [
            "customer_unique_id",
            "order_purchase_timestamp"
        ]
    ]
    .rename(
        columns={
            "order_purchase_timestamp": "second_purchase_ts"
        }
    )
    .copy()
)

In [62]:
# merge 回 first_purchase
first_purchase = first_purchase.merge(
    second_purchase,
    on="customer_unique_id",
    how="left"
)

In [63]:
first_purchase

,customer_unique_id,first_purchase_ts,delivery_days_mean,first_event_review_score_mean,late_flag,has_review,customer_state,purchase_event_rank,second_purchase_ts
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,6.411111,5.0,0,1,SP,1,NaT
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:00,3.285417,4.0,0,1,SP,1,NaT
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,25.731250,3.0,0,1,SC,1,NaT
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:00,20.037500,4.0,0,1,PA,1,NaT
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,13.140972,5.0,0,1,SP,1,NaT
...,...,...,...,...,...,...,...,...,...
93345,fffcf5a5ff07b0908bd4e2dbc735a684,2017-06-08 21:00:00,27.515278,5.0,0,1,PE,1,NaT
93346,fffea47cd6d3cc0a88bd621562a9d061,2017-12-10 20:07:00,30.097917,4.0,0,1,BA,1,NaT
93347,ffff371b4d645b6ecea244b27531430a,2017-02-07 15:49:00,14.872222,5.0,0,1,MT,1,NaT
93348,ffff5962728ec6157033ef9805bacc48,2018-05-02 15:17:00,11.859028,5.0,0,1,ES,1,NaT


In [65]:
# 建立90天內是否二購label 
first_purchase["days_to_second_purchase"] = (
    first_purchase["second_purchase_ts"]
    - first_purchase["first_purchase_ts"]
).dt.days

first_purchase["purchase_2nd_within_90d"] = (
    first_purchase["days_to_second_purchase"].between(0, 90)
).astype(int)



purchase_2nd_within_90d
0    92084
1     1266
Name: count, dtype: int64

In [66]:
first_purchase["purchase_2nd_within_90d"].value_counts()

purchase_2nd_within_90d
0    92084
1     1266
Name: count, dtype: int64

In [67]:
first_purchase.isna().sum()

customer_unique_id                   0
first_purchase_ts                    0
delivery_days_mean                   0
first_event_review_score_mean      614
late_flag                            0
has_review                           0
customer_state                       0
purchase_event_rank                  0
second_purchase_ts               91181
days_to_second_purchase          91181
purchase_2nd_within_90d              0
dtype: int64

In [69]:
# mean_review 缺失補中位數 (數值欄位補值+缺失訊號保留)
# 計算中位數
review_median = first_purchase[
    "first_event_review_score_mean"
].median()

print(review_median)

# 補中位數
first_purchase[
    "first_event_review_score_mean"
] = first_purchase[
    "first_event_review_score_mean"
].fillna(review_median)

5.0


In [70]:
first_purchase.isna().sum()

customer_unique_id                   0
first_purchase_ts                    0
delivery_days_mean                   0
first_event_review_score_mean        0
late_flag                            0
has_review                           0
customer_state                       0
purchase_event_rank                  0
second_purchase_ts               91181
days_to_second_purchase          91181
purchase_2nd_within_90d              0
dtype: int64

In [74]:
# 截斷資料後面90天的資料(只保留首購後有完整 90 天觀察期的顧客)
data_end = purchase_event_base["order_purchase_timestamp"].max()
analysis_end = data_end - pd.Timedelta(days=90)

first_purchase = first_purchase[
    first_purchase["first_purchase_ts"] <= analysis_end
].copy()

In [75]:
# 刪除不進模型的欄位 
secondary_purchase_model_abt = first_purchase.drop(
    columns=[
        "purchase_event_rank",
        "second_purchase_ts",
        "days_to_second_purchase"
    ]
).copy()

In [76]:
# 確認label 分佈 (90 天二購率 = 1.483%。)
secondary_purchase_model_abt["purchase_2nd_within_90d"].value_counts(normalize=True)

purchase_2nd_within_90d
0    0.98517
1    0.01483
Name: proportion, dtype: float64

In [79]:
secondary_purchase_model_abt

,customer_unique_id,first_purchase_ts,delivery_days_mean,first_event_review_score_mean,late_flag,has_review,customer_state,purchase_2nd_within_90d
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,6.411111,5.0,0,1,SP,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:00,3.285417,4.0,0,1,SP,0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,25.731250,3.0,0,1,SC,0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:00,20.037500,4.0,0,1,PA,0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,13.140972,5.0,0,1,SP,0
...,...,...,...,...,...,...,...,...
93345,fffcf5a5ff07b0908bd4e2dbc735a684,2017-06-08 21:00:00,27.515278,5.0,0,1,PE,0
93346,fffea47cd6d3cc0a88bd621562a9d061,2017-12-10 20:07:00,30.097917,4.0,0,1,BA,0
93347,ffff371b4d645b6ecea244b27531430a,2017-02-07 15:49:00,14.872222,5.0,0,1,MT,0
93348,ffff5962728ec6157033ef9805bacc48,2018-05-02 15:17:00,11.859028,5.0,0,1,ES,0


In [78]:
# 存檔
secondary_purchase_model_abt.to_csv(
    "/Users/kaiping/Desktop/olist_project/data/processed/abt_2nd_purchase_v1.csv",
    index=False
)